# Tạo SRT tiếng Trung từ WAV trên Colab GPU

Upload WAV vào `/content`, sửa `AUDIO`, rồi chạy các ô theo thứ tự. WAV phải giữ nguyên khoảng lặng và bắt đầu cùng mốc 0 với video.

Whisper lấy lời nói; align và refine ước lượng mốc của từ/câu trên chính WAV; VAD tìm vùng tiếng nói chưa có chữ. Những câu nhận diện lại chỉ được thêm nếu đạt tiêu chí độ tin cậy và không chồng câu khác.

Kết quả chính là một file SRT và Colab tải trực tiếp file đó. `report.json` ghi câu cần nghe lại; `review_audio` chứa âm thanh của vùng nghi thiếu lời nói. Không thuật toán nào bảo đảm chính xác tuyệt đối khi tiếng nhỏ, chồng giọng hoặc có nhạc.

In [ ]:
%pip install -q stable-ts silero-vad soundfile scipy


Nếu Colab yêu cầu khởi động lại phiên sau khi cài, hãy khởi động lại rồi chạy tiếp từ ô cấu hình. Chọn GPU: Runtime → Change runtime type → T4 GPU.

WAV cần được trích từ chính video bạn sẽ dịch và giữ nguyên khoảng lặng.

In [ ]:
AUDIO = "/content/nguoi_vo_tam_part001.wav"
OUTPUT_ROOT = "/content/srt_results"
MODEL = "large-v3"
LANGUAGE = "zh"
INITIAL_PROMPT = ""  # Chỉ điền tên riêng đã biết chắc; không điền lời thoại phỏng đoán.
SPLIT_GAP = 0.28  # Có thể thử 0.35 nếu câu bị chia quá nhỏ.
ENABLE_ALIGNMENT = True
ENABLE_REFINE = True  # Tốn thêm thời gian; xử lý từng batch để giảm VRAM.
AUDIT_MISSING = True
MAX_AUDIT_CLIPS = 30  # Lấy các vùng nghi bỏ sót dài nhất; báo cáo giữ danh sách đầy đủ.


In [ ]:
"""Dán toàn bộ file vào MỘT ô Colab rồi Run; chọn GPU trước khi chạy.

Code cài thư viện, đọc WAV (hoặc mở hộp upload), tạo SRT và tải trực tiếp SRT.
Sửa AUDIO ở cuối file nếu WAV của bạn có tên khác.
"""
from __future__ import annotations


import copy
import gc
import importlib.metadata
import json
import math
import re
from datetime import datetime
from math import gcd
from pathlib import Path

import numpy as np
import soundfile as sf
import stable_whisper
import torch
from scipy.signal import resample_poly
from silero_vad import get_speech_timestamps, load_silero_vad


def normalized(text):
    return re.sub(r"\s+", "", text)


def full_text(result):
    return "".join(segment.text for segment in result.segments)


def dump_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")


def read_audio(path):
    audio, rate = sf.read(path, dtype="float32")
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    if rate != 16000:
        divisor = gcd(rate, 16000)
        audio = resample_poly(audio, 16000 // divisor, rate // divisor)
    audio = np.ascontiguousarray(audio, dtype=np.float32)
    if not len(audio) or not np.isfinite(audio).all():
        raise ValueError("WAV rỗng hoặc có mẫu âm thanh không hợp lệ.")
    return audio


def stamp(seconds):
    milliseconds = max(0, round(seconds * 1000))
    hours, milliseconds = divmod(milliseconds, 3600000)
    minutes, milliseconds = divmod(milliseconds, 60000)
    seconds, milliseconds = divmod(milliseconds, 1000)
    return f"{hours:02}:{minutes:02}:{seconds:02},{milliseconds:03}"


def cues_from(result, duration):
    cues, issues = [], []
    for segment in result.segments:
        text = segment.text.strip()
        if not text:
            continue
        index = len(cues) + 1
        start, end = float(segment.start), float(segment.end)
        if not (math.isfinite(start) and math.isfinite(end)):
            raise ValueError(f"Câu {index} có mốc không hữu hạn; kiểm tra JSON nhận diện.")
        if start < 0 or end > duration or end <= start:
            issues.append({"segment": index, "reason": "invalid_or_zero_duration",
                           "start": start, "end": end})
            start = min(max(0.0, start), max(0.0, duration - 0.04))
            end = min(duration, max(start + 0.04, end))
        if cues and start < cues[-1]["end"] - 0.02:
            issues.append({"segment": index, "reason": "overlap_or_wrong_order",
                           "seconds": round(cues[-1]["end"] - start, 3)})
        if end - start > 10 or len(normalized(text)) > 60:
            issues.append({"segment": index, "reason": "long_cue_review_speaker_turns"})
        probabilities = [float(w.probability) for w in (segment.words or [])
                         if w.probability is not None]
        if probabilities and sum(probabilities) / len(probabilities) < 0.55:
            issues.append({"segment": index, "reason": "low_word_probability"})
        cues.append({"id": index, "start": start, "end": end, "text": text})
    return cues, issues


def write_srt(path, cues):
    Path(path).write_text("\n\n".join(
        f"{cue['id']}\n{stamp(cue['start'])} --> {stamp(cue['end'])}\n{cue['text']}"
        for cue in cues
    ) + "\n", encoding="utf-8")


def word_spans(result, duration, margin=0.12):
    spans = []
    for segment in result.segments:
        for word in segment.words or [segment]:
            spans.append((max(0.0, float(word.start) - margin),
                          min(duration, float(word.end) + margin)))
    merged = []
    for start, end in sorted(spans):
        if end <= start:
            continue
        if merged and start <= merged[-1][1]:
            merged[-1][1] = max(end, merged[-1][1])
        else:
            merged.append([start, end])
    return merged


def uncovered_speech(speech, covered, minimum=0.55):
    """Find VAD intervals outside word timestamps, keeping source audio time."""
    missing = []
    for region in speech:
        cursor, end = region["start"], region["end"]
        for left, right in covered:
            if right <= cursor:
                continue
            if left >= end:
                break
            if left - cursor >= minimum:
                missing.append({"start": cursor, "end": min(left, end)})
            cursor = max(cursor, right)
            if cursor >= end:
                break
        if end - cursor >= minimum:
            missing.append({"start": cursor, "end": end})
    return missing


def recovery_cue(retry, gap, clip_start, existing):
    """Accept a second-pass phrase only when its words sit inside an empty speech gap."""
    words = [word for segment in retry.segments for word in (segment.words or [])
             if gap["start"] <= clip_start + (word.start + word.end) / 2 <= gap["end"]]
    if not words or any(word.probability is None for word in words):
        return None, "missing_word_confidence"
    probabilities = [float(word.probability) for word in words]
    if min(probabilities) < 0.50 or sum(probabilities) / len(probabilities) < 0.75:
        return None, "low_word_confidence"
    text = "".join(word.word for word in words).strip()
    if len(normalized(text)) < 2:
        return None, "short_text_requires_listening"
    start = max(gap["start"], clip_start + float(words[0].start))
    end = min(gap["end"], clip_start + float(words[-1].end))
    if end - start < 0.12:
        return None, "invalid_or_short_timing"
    nearby = [cue for cue in existing if cue["end"] >= start - 2 and cue["start"] <= end + 2]
    if any(cue["start"] < end + 0.03 and cue["end"] > start - 0.03 for cue in nearby):
        return None, "overlaps_existing_sentence"
    if any(normalized(text) in normalized(cue["text"]) for cue in nearby):
        return None, "repeats_nearby_text"
    return {"start": start, "end": end, "text": text, "source": "recovered_speech"}, "accepted"


def run_srt(audio_path, output_root="/content/srt_results", *, model_name="large-v3",
            language="zh", initial_prompt="", split_gap=0.28,
            enable_alignment=True, enable_refine=True, audit_missing=True,
            max_audit_clips=30):
    """Return paths and cues; this function never modifies the input WAV.

    Keep first-pass text. Add independently recognized speech gaps only when
    word confidence, placement, uniqueness and non-overlap checks all pass.
    """
    audio_path = Path(audio_path)
    if not audio_path.is_file():
        raise FileNotFoundError(f"Không tìm thấy {audio_path}; upload WAV hoặc sửa AUDIO.")
    if not torch.cuda.is_available():
        raise RuntimeError("Chọn Runtime > Change runtime type > GPU trên Colab.")
    if split_gap <= 0:
        raise ValueError("split_gap phải lớn hơn 0.")
    if max_audit_clips < 0:
        raise ValueError("max_audit_clips không được âm.")
    gc.collect()
    torch.cuda.empty_cache()
    print("GPU:", torch.cuda.get_device_name(0))
    audio = read_audio(audio_path)
    duration = len(audio) / 16000
    out = Path(output_root) / (audio_path.stem + "-" + datetime.now().strftime("%Y%m%d-%H%M%S-%f"))
    out.mkdir(parents=True, exist_ok=False)
    report = {
        "audio": str(audio_path), "duration_seconds": duration,
        "model": model_name, "language": language,
        "settings": {"split_gap": split_gap, "enable_alignment": enable_alignment,
                     "enable_refine": enable_refine, "audit_missing": audit_missing,
                     "max_audit_clips": max_audit_clips, "initial_prompt": initial_prompt},
        "versions": {name: importlib.metadata.version(name)
                     for name in ("stable-ts", "openai-whisper", "silero-vad", "torch")},
        "warnings": [],
    }
    dump_json(out / "report.json", report)
    print(f"WAV dài {duration:.2f}s. Đầu ra: {out}")
    print("1/5 — Nhận diện toàn bộ WAV, giữ nguyên khoảng lặng...")
    # CPU preload holds both checkpoint and model weights in host RAM while
    # loading. Colab's host RAM can run out before the model reaches the GPU.
    print("Đang tải model với cpu_preload=False...", flush=True)
    model = stable_whisper.load_model(
        model_name, device="cuda", cpu_preload=False, in_memory=False,
    )
    model.eval()
    gc.collect()
    print(f"Model đã lên GPU: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB VRAM", flush=True)
    decode = dict(language=language, task="transcribe", word_timestamps=True,
                  regroup=False, suppress_silence=False, temperature=0.0,
                  beam_size=5, fp16=True, verbose=False, max_instant_words=1.0)
    raw = model.transcribe(audio, condition_on_previous_text=True,
                           initial_prompt=initial_prompt or None, **decode)
    if not raw.segments:
        raise RuntimeError("Whisper không nhận được text; kiểm tra WAV và ngôn ngữ.")
    raw.save_as_json(str(out / "01_asr_raw.json"))
    reference_text = normalized(full_text(raw))
    (out / "transcript.txt").write_text(full_text(raw).strip(), encoding="utf-8")
    work = copy.deepcopy(raw)

    print("2/5 — Ước lượng lại mốc từng từ theo WAV...")
    if enable_alignment:
        try:
            candidate = model.align(audio, copy.deepcopy(raw), language=language,
                                    original_split=True, regroup=False,
                                    remove_instant_words=False, suppress_silence=False,
                                    nonspeech_skip=None, fast_mode=False, verbose=False)
            if candidate is None or normalized(full_text(candidate)) != reference_text:
                raise ValueError("Alignment làm thay đổi text; dùng lại bản ASR.")
            work = candidate
        except Exception as exc:
            report["warnings"].append({"stage": "align", "fallback": "raw_asr", "error": str(exc)})
            print("CẢNH BÁO align:", exc)
            torch.cuda.empty_cache()
    # Only split existing segments. No merging across ASR segment boundaries.
    work.split_by_gap(split_gap)
    work.split_by_punctuation(["。", "？", "！", "?", "!"])
    if normalized(full_text(work)) != reference_text:
        raise RuntimeError("Text đổi sau khi chia câu; xem 01_asr_raw.json.")
    work.save_as_json(str(out / "02_aligned.json"))

    print("3/5 — Tinh chỉnh đầu/cuối từng câu...")
    if enable_refine:
        try:
            refined = model.refine(audio, copy.deepcopy(work), steps="se",
                                   word_level=False, precision=0.04, single_batch=True,
                                   abs_dur_change=0.30, inplace=True, verbose=False)
            if normalized(full_text(refined)) != reference_text:
                raise ValueError("Refine làm thay đổi text; giữ bản trước refine.")
            work = refined
        except Exception as exc:
            report["warnings"].append({"stage": "refine", "fallback": "before_refine", "error": str(exc)})
            print("CẢNH BÁO refine:", exc)
            torch.cuda.empty_cache()
    work.save_as_json(str(out / "03_final_words.json"))
    cues, issues = cues_from(work, duration)
    if normalized("".join(c["text"] for c in cues)) != reference_text:
        raise RuntimeError("Text xuất SRT khác bản ASR; xem JSON trước khi dùng.")
    srt = out / f"{audio_path.stem}.srt"

    print("4/5 — VAD kiểm tra vùng chưa có text; không kéo START về đầu vùng VAD...")
    vad_model = load_silero_vad()
    regions = get_speech_timestamps(torch.from_numpy(audio), vad_model,
                                   sampling_rate=16000, threshold=0.35,
                                   min_silence_duration_ms=160, speech_pad_ms=0,
                                   return_seconds=False)
    # Use exact sample indices. return_seconds=True defaults to coarse rounding.
    speech = [{"start": r["start"] / 16000, "end": r["end"] / 16000} for r in regions]
    missing = uncovered_speech(speech, word_spans(work, duration))
    for cue in cues:
        overlaps = [r for r in speech if r["end"] > cue["start"] and r["start"] < cue["end"]]
        if not overlaps:
            issues.append({"segment": cue["id"], "reason": "no_vad_overlap_listen_before_removing"})
        elif overlaps[0]["start"] - cue["start"] > 0.20:
            issues.append({"segment": cue["id"], "reason": "starts_before_vad_review",
                           "seconds": round(overlaps[0]["start"] - cue["start"], 3)})

    print(f"5/5 — Nhận diện lại vùng nghi bỏ sót: {len(missing)} vùng...")
    review_clips = out / "review_audio"
    review_clips.mkdir()
    selected = sorted(missing, key=lambda r: r["end"] - r["start"], reverse=True)[:max_audit_clips]
    recovery = []
    recovered = []
    if audit_missing:
        for index, gap in enumerate(selected, 1):
            # Each audit clip contains at most 12s core speech plus surrounding context.
            end_core = min(gap["end"], gap["start"] + 12)
            left = max(0, int((gap["start"] - 0.45) * 16000))
            right = min(len(audio), int((end_core + 0.45) * 16000))
            clip = audio[left:right]
            clip_name = f"gap_{index:03d}_{left / 16000:.3f}s.wav"
            sf.write(review_clips / clip_name, clip, 16000)
            entry = {**gap, "clip_start": left / 16000, "clip_end": right / 16000,
                     "clip": f"review_audio/{clip_name}", "candidate_text": "",
                     "added_to_srt": False}
            try:
                retry = model.transcribe(clip, condition_on_previous_text=False, **decode)
                entry["candidate_text"] = full_text(retry).strip()
                suggestion, reason = recovery_cue(retry, gap, left / 16000, cues + recovered)
                entry["decision"] = reason
                if suggestion is not None:
                    recovered.append(suggestion)
                    entry["added_to_srt"] = True
                    entry["accepted_text"] = suggestion["text"]
                    entry["accepted_start"] = suggestion["start"]
                    entry["accepted_end"] = suggestion["end"]
                else:
                    entry["note"] = "Nghe clip nếu muốn kiểm tra chữ bị sót; chưa thêm vào SRT."
            except Exception as exc:
                entry["error"] = str(exc)
                torch.cuda.empty_cache()
            recovery.append(entry)
            print(f"  {index}/{len(selected)}: {stamp(gap['start'])} — {entry['candidate_text']}")

    cues.extend(recovered)
    cues.sort(key=lambda cue: (cue["start"], cue["end"]))
    old_to_new = {cue["id"]: index for index, cue in enumerate(cues, 1) if "id" in cue}
    for index, cue in enumerate(cues, 1):
        cue["id"] = index
    for issue in issues:
        issue["segment"] = old_to_new.get(issue["segment"], issue["segment"])
    write_srt(srt, cues)

    report.update(
        raw_segments=len(raw.segments), final_segments=len(cues),
        alignment_applied=enable_alignment and not any(w["stage"] == "align" for w in report["warnings"]),
        refinement_applied=enable_refine and not any(w["stage"] == "refine" for w in report["warnings"]),
        first_pass_text_preserved=normalized(full_text(work)) == reference_text,
        recovered_segments=len(recovered),
        text_note="Câu thêm tự động đã qua lọc xác suất và timeline; nghe lại vẫn cần thiết khi âm thanh khó.",
        timeline_issues=issues, vad_regions=speech, uncovered_speech=missing,
        recovery_candidates=recovery,
        unaudited_regions=len(missing) - len(recovery),
    )
    dump_json(out / "report.json", report)
    (out / "recovery_candidates.txt").write_text("\n\n".join(
        f"{stamp(r['start'])} --> {stamp(r['end'])}\n{r['candidate_text']}\n{r['clip']}"
        for r in recovery
    ), encoding="utf-8")
    print(f"\nSRT: {srt}\nCâu cần xem: {len(set(i['segment'] for i in issues))}")
    print(f"Giữ text lượt đầu: True | Câu bổ sung: {len(recovered)} | Alignment: {report['alignment_applied']} | Refine: {report['refinement_applied']}")
    print(f"Báo cáo: {out / 'report.json'}")
    return {"srt": str(srt), "folder": str(out), "cues": cues, "report": report}




In [ ]:
result = run_srt(
    AUDIO, OUTPUT_ROOT, model_name=MODEL, language=LANGUAGE,
    initial_prompt=INITIAL_PROMPT, split_gap=SPLIT_GAP,
    enable_alignment=ENABLE_ALIGNMENT, enable_refine=ENABLE_REFINE,
    audit_missing=AUDIT_MISSING, max_audit_clips=MAX_AUDIT_CLIPS,
)


Colab tải trực tiếp file SRT ở ô bên dưới. `report.json` và các clip cần nghe lại vẫn nằm trong `/content/srt_results` trên Colab. Một câu được thêm từ vùng VAD sẽ có `added_to_srt: true` trong báo cáo. Những vùng chưa đạt tiêu chí không được tự thêm vào SRT.

In [ ]:
from google.colab import files
files.download(result["srt"])


Ô tùy chọn dưới đây phát giọng gốc quanh một câu trong SRT mới. `CUE_ID` là số câu trong SRT mới.

In [ ]:
from IPython.display import Audio, display
CUE_ID = 7
cue = next(c for c in result["cues"] if c["id"] == CUE_ID)
wave = read_audio(AUDIO)
left = max(0, int((cue["start"] - 0.5) * 16000))
right = min(len(wave), int((cue["end"] + 0.5) * 16000))
print(cue)
print(f'START trong clip nghe: {cue["start"] - left / 16000:.3f}s')
display(Audio(wave[left:right], rate=16000))


Tài liệu API: [stable-ts alignment/refinement](https://github.com/jianfch/stable-ts#alignment), [Silero timestamp implementation](https://github.com/snakers4/silero-vad/blob/master/src/silero_vad/utils_vad.py). Notebook đã được kiểm tra cú pháp tại máy phát triển; chưa chạy suy luận trên Colab/T4.